# STAIR-v2: Modality-Adaptive kNN Graph Reweighting (STAIR-DyFuse)

**Core idea:** Per-item L2-norm confidence score reweights textual + visual kNN graph edges.  
**Key property:** 0 extra parameters — only changes `mAdj` construction in `prepare()`.  
**Baseline papers:** TAMER (weighted graph fusion) + NLGCL+ (norm-based confidence signal)

| | |
|---|---|
| Dataset | Amazon2014Baby + Amazon2014Sports |
| Epochs | 500 |
| Embedding dim | 64 |
| Optimizer | AdamWSEvo |
| Confidence mode | norm (linear L2-norm ratio) |

In [ ]:
import os, shutil

os.chdir('/kaggle/working')

# Clone / refresh repo
repo = 'STAIR-Enhanced'
if os.path.exists(repo):
    shutil.rmtree(repo)
os.system('git clone https://github.com/ThanhChuong12/STAIR-Enhanced.git')

# Install dependencies
os.system('pip install nvidia-ml-py -q')
os.system('pip install torchdata==0.6.1 --no-deps -q')
os.system('pip install freerec==0.9.7 -q')
os.system('pip install torch_geometric -q')
os.system('pip install prettytable -q')

import torch, platform, freerec
print('Python  :', platform.python_version())
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('freerec :', freerec.__version__)
print('Environment ready')


In [ ]:
# Cell 2: Copy dataset files from /kaggle/input to /kaggle/data
import os, shutil

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)

def copy_dataset(keywords, full_name):
    dest = os.path.join(DATA_ROOT, full_name)
    os.makedirs(dest, exist_ok=True)
    copied = []
    for root, _, files in os.walk('/kaggle/input'):
        if any(kw.lower() in root.lower() for kw in keywords):
            for f in files:
                if f.endswith(('.npy', '.pkl', '.txt', '.inter', '.item')):
                    shutil.copy(os.path.join(root, f), os.path.join(dest, f))
                    copied.append(f)
    print(f'[{full_name}] {len(copied)} files copied')

copy_dataset(['baby', 'amazon2014baby'],   'Amazon2014Baby_550_MMRec')
copy_dataset(['sports', 'amazon2014sports'], 'Amazon2014Sports_550_MMRec')
print('Data ready at', DATA_ROOT)


In [ ]:
# Cell 3: Modality Confidence Score Diagnostics
import os, sys, pickle, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, '/kaggle/working/STAIR-Enhanced')
warnings.filterwarnings('ignore', category=RuntimeWarning)

DATA_ROOT = '/kaggle/data'
DATASETS  = ['Amazon2014Baby_550_MMRec', 'Amazon2014Sports_550_MMRec']
MFILES    = ['textual_modality.pkl', 'visual_modality.pkl']

def load_feat(ds, mf):
    with open(os.path.join(DATA_ROOT, ds, mf), 'rb') as f:
        return pickle.load(f)

def compute_confidence_std(ft, fv):
    eps = 1e-7
    std_v = fv.std(dim=0).mean()
    std_t = ft.std(dim=0).mean()
    global_cv = std_v / (std_v + std_t + eps)
    c_v = torch.full((fv.shape[0],), global_cv.item())
    return c_v, 1.0 - c_v, std_v.item(), std_t.item()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('STAIR-DyFuse -- Modality Confidence Diagnostics', fontsize=14, fontweight='bold')

for ri, ds in enumerate(DATASETS):
    ds_name = 'Baby' if 'Baby' in ds else 'Sports'
    try:
        ft = load_feat(ds, MFILES[0])
        fv = load_feat(ds, MFILES[1])
    except FileNotFoundError:
        print(f'[WARN] Files not found for {ds_name}. Run Cell 2 first.'); continue

    ft = torch.tensor(ft, dtype=torch.float32) if not isinstance(ft, torch.Tensor) else ft.float()
    fv = torch.tensor(fv, dtype=torch.float32) if not isinstance(fv, torch.Tensor) else fv.float()

    nt = ft.norm(p=2, dim=-1).numpy()
    nv = fv.norm(p=2, dim=-1).numpy()
    cv, ct, std_v, std_t = compute_confidence_std(ft, fv)
    cv_val = cv[0].item()

    print(f'\n[{ds_name}] Text  norm: mean={nt.mean():.4f}, std={nt.std():.4f} | feature std={std_t:.4f}')
    print(f'[{ds_name}] Visual norm: mean={nv.mean():.4f}, std={nv.std():.4f} | feature std={std_v:.4f}')
    print(f'[{ds_name}] Global confidence: c_v={cv_val:.4f}, c_t={1-cv_val:.4f}')
    print(f'[{ds_name}] => Visual modality weighted {cv_val*100:.1f}%, Text weighted {(1-cv_val)*100:.1f}%')

    ax = axes[ri]

    # --- Plot 1: Visual L2-norm distribution (text is pre-normalized to 1.0) ---
    ax[0].hist(nv, bins=60, color='#E8734A', alpha=0.85, label='Visual L2-norm', density=True)
    ax[0].axvline(nt.mean(), color='#4A90D9', linewidth=2.5, linestyle='--',
                  label=f'Text L2-norm = {nt.mean():.3f}\n(pre-normalized)')
    ax[0].set_title(f'{ds_name} -- L2-Norm Distribution', fontweight='bold')
    ax[0].set_xlabel('L2 Norm'); ax[0].set_ylabel('Density')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

    # --- Plot 2: Confidence breakdown (std-based) ---
    bars = ax[1].bar(['Text\n(std-based)', 'Visual\n(std-based)'],
                     [1 - cv_val, cv_val],
                     color=['#4A90D9', '#E8734A'], alpha=0.85, edgecolor='white', width=0.5)
    ax[1].set_title(f'{ds_name} -- Modality Confidence Weight', fontweight='bold')
    ax[1].set_ylabel('Confidence Score'); ax[1].set_ylim(0, 1.15)
    for bar, val, lbl in zip(bars, [1-cv_val, cv_val],
                              [f'c_t = {1-cv_val:.3f}', f'c_v = {cv_val:.3f}']):
        ax[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                   lbl, ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax[1].text(0.5, 1.05,
               f'feature_std_v={std_v:.4f}  |  feature_std_t={std_t:.4f}',
               ha='center', va='center', transform=ax[1].transAxes,
               fontsize=8, color='gray', style='italic')
    ax[1].grid(alpha=0.3, axis='y')

    # --- Plot 3: Per-dim std distribution (text vs visual, per feature dimension) ---
    dim_std_t = ft.std(dim=0).numpy()   # std of each text feature dim across items
    dim_std_v = fv.std(dim=0).numpy()   # std of each visual feature dim across items
    ax[2].hist(dim_std_t, bins=50, alpha=0.7, color='#4A90D9', label='Text dim-std', density=True)
    ax[2].hist(dim_std_v, bins=50, alpha=0.7, color='#E8734A', label='Visual dim-std', density=True)
    ax[2].axvline(std_t, color='#2471A3', linewidth=2, linestyle='--',
                  label=f'Text mean={std_t:.4f}')
    ax[2].axvline(std_v, color='#A04000', linewidth=2, linestyle='--',
                  label=f'Visual mean={std_v:.4f}')
    ax[2].set_title(f'{ds_name} -- Per-Dim Feature Std', fontweight='bold')
    ax[2].set_xlabel('Std across items (per dim)'); ax[2].set_ylabel('Density')
    ax[2].legend(fontsize=7); ax[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v2_confidence_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNote: Confidence uses std-based scoring (main_v2.py compute_confidence).')
print('Text features are pre-L2-normalized -> norm=1.0, std>0 per dim -> std-based scoring is correct.')


In [ ]:
# Cell 4: Train STAIR-DyFuse -- Amazon Baby
import subprocess, time, os

os.chdir('/kaggle/working/STAIR-Enhanced')
LOG_BABY_V2 = '/kaggle/working/log_stair_v2_baby.txt'

cmd = (
    'python main_v2.py '
    '--root /kaggle/data '
    '--dataset Amazon2014Baby_550_MMRec '
    '--epochs 500 --batch-size 1024 --embedding-dim 64 '
    '--num-layers 3 --num-neighbors 5-1 '
    '--conf-mode norm --conf-temp 1.0 '
    '--optimizer adamwsevo --lr 1e-3 --weight-decay 0.1 --seed 1 '
    f'> {LOG_BABY_V2} 2>&1'
)
print('Training STAIR-DyFuse on Baby...')
t0 = time.time()
ret = subprocess.run(cmd, shell=True)
print(f'Done in {(time.time()-t0)/60:.1f} min | rc={ret.returncode}')
if ret.returncode != 0:
    with open(LOG_BABY_V2, 'r', errors='ignore') as f: print(f.read()[-3000:])


In [ ]:
# Cell 5: Train STAIR-DyFuse -- Amazon Sports
import subprocess, time, os

os.chdir('/kaggle/working/STAIR-Enhanced')
LOG_SPORTS_V2 = '/kaggle/working/log_stair_v2_sports.txt'

cmd = (
    'python main_v2.py '
    '--root /kaggle/data '
    '--dataset Amazon2014Sports_550_MMRec '
    '--epochs 500 --batch-size 1024 --embedding-dim 64 '
    '--num-layers 3 --num-neighbors 5-1 '
    '--conf-mode norm --conf-temp 1.0 '
    '--optimizer adamwsevo --lr 1e-3 --weight-decay 0.1 --seed 1 '
    f'> {LOG_SPORTS_V2} 2>&1'
)
print('Training STAIR-DyFuse on Sports...')
t0 = time.time()
ret = subprocess.run(cmd, shell=True)
print(f'Done in {(time.time()-t0)/60:.1f} min | rc={ret.returncode}')
if ret.returncode != 0:
    with open(LOG_SPORTS_V2, 'r', errors='ignore') as f: print(f.read()[-3000:])


In [ ]:
import re, os

LOG_BABY_V2   = '/kaggle/working/log_stair_v2_baby.txt'
LOG_SPORTS_V2 = '/kaggle/working/log_stair_v2_sports.txt'

def parse_log(log_path):
    """Parse freerec training log: returns (train_loss, val_metrics, best_metrics)."""
    if not os.path.exists(log_path):
        print(f'[WARN] Log not found: {log_path}')
        return [], {m: [] for m in ['Recall@10','Recall@20','NDCG@10','NDCG@20']}, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    train_loss = []
    val_metrics = {'Recall@10': [], 'Recall@20': [], 'NDCG@10': [], 'NDCG@20': []}
    MPATS = {
        'Recall@10': re.compile(r'recall@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'Recall@20': re.compile(r'recall@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'NDCG@10':   re.compile(r'ndcg@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
        'NDCG@20':   re.compile(r'ndcg@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
    }
    LPAT = re.compile(r'loss\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I)
    for line in lines:
        m = LPAT.search(line)
        if m and 'train' in line.lower():
            train_loss.append(float(m.group(1)))
        for metric, pat in MPATS.items():
            mm = pat.search(line)
            if mm:
                val_metrics[metric].append(float(mm.group(1)))
    best = {m: max(v) for m, v in val_metrics.items() if v}
    return train_loss, val_metrics, best

print('Parsing Baby V2...')
loss_baby_v2, val_baby_v2, best_baby_v2 = parse_log(LOG_BABY_V2)
print(f'  loss={len(loss_baby_v2)} epochs, val={len(val_baby_v2["Recall@10"])} epochs')
print(f'  best={best_baby_v2}')
print('Parsing Sports V2...')
loss_sports_v2, val_sports_v2, best_sports_v2 = parse_log(LOG_SPORTS_V2)
print(f'  loss={len(loss_sports_v2)} epochs, val={len(val_sports_v2["Recall@10"])} epochs')
print(f'  best={best_sports_v2}')

# Learning Curves
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def smooth(v, w=10):
    return np.convolve(v, np.ones(w)/w, mode='valid') if len(v) >= w else v

CLRS = {'train':'#E74C3C','r10':'#2ECC71','r20':'#27AE60','n10':'#3498DB','n20':'#2980B9','sm':'#F39C12'}

fig = plt.figure(figsize=(20, 13))
fig.suptitle('STAIR-DyFuse (v2) -- Learning Curves', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.38)

datasets_info = [
    ('Baby',   loss_baby_v2,   val_baby_v2),
    ('Sports', loss_sports_v2, val_sports_v2),
]

for ci, (ds_name, t_loss, val_m) in enumerate(datasets_info):
    cb = ci * 2
    ep_l = range(1, len(t_loss)+1)
    ep_v = range(1, len(val_m['Recall@10'])+1)

    ax = fig.add_subplot(gs[0, cb:cb+2])
    ax.plot(ep_l, t_loss, color=CLRS['train'], alpha=0.4, lw=0.8)
    if len(t_loss) >= 10:
        s = smooth(t_loss)
        ax.plot(range(5, 5+len(s)), s, color=CLRS['sm'], lw=2, label='Smoothed (w=10)')
    ax.set_title(f'{ds_name} -- BPR Training Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('BPR Loss')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    ax = fig.add_subplot(gs[1, cb:cb+2])
    if val_m['Recall@10']:
        ax.plot(ep_v, val_m['Recall@10'], color=CLRS['r10'], lw=1.8,
                label='Recall@10', marker='o', ms=2, markevery=20)
    if val_m['Recall@20']:
        ax.plot(ep_v, val_m['Recall@20'], color=CLRS['r20'], lw=1.8,
                label='Recall@20', marker='s', ms=2, markevery=20)
    ax.set_title(f'{ds_name} -- Recall Validation Curve', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Recall')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    ax = fig.add_subplot(gs[2, cb:cb+2])
    if val_m['NDCG@10']:
        ax.plot(ep_v, val_m['NDCG@10'], color=CLRS['n10'], lw=1.8,
                label='NDCG@10', marker='^', ms=2, markevery=20)
    if val_m['NDCG@20']:
        ax.plot(ep_v, val_m['NDCG@20'], color=CLRS['n20'], lw=1.8,
                label='NDCG@20', marker='v', ms=2, markevery=20)
    ax.set_title(f'{ds_name} -- NDCG Validation Curve', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('NDCG')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v2_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curves saved.')


In [ ]:
import re, os

LOG_BABY_V2   = '/kaggle/working/log_stair_v2_baby.txt'
LOG_SPORTS_V2 = '/kaggle/working/log_stair_v2_sports.txt'

def parse_log(log_path):
    """Parse freerec training log: returns (train_loss, val_metrics, best_metrics)."""
    if not os.path.exists(log_path):
        print(f'[WARN] Log not found: {log_path}')
        return [], {m: [] for m in ['Recall@10','Recall@20','NDCG@10','NDCG@20']}, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    train_loss = []
    val_metrics = {'Recall@10': [], 'Recall@20': [], 'NDCG@10': [], 'NDCG@20': []}
    MPATS = {
        'Recall@10': re.compile(r'recall@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'Recall@20': re.compile(r'recall@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'NDCG@10':   re.compile(r'ndcg@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
        'NDCG@20':   re.compile(r'ndcg@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
    }
    LPAT = re.compile(r'loss\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I)
    for line in lines:
        m = LPAT.search(line)
        if m and 'train' in line.lower():
            train_loss.append(float(m.group(1)))
        for metric, pat in MPATS.items():
            mm = pat.search(line)
            if mm:
                val_metrics[metric].append(float(mm.group(1)))
    best = {m: max(v) for m, v in val_metrics.items() if v}
    return train_loss, val_metrics, best

print('Parsing Baby V2...')
loss_baby_v2, val_baby_v2, best_baby_v2 = parse_log(LOG_BABY_V2)
print(f'  loss={len(loss_baby_v2)} epochs, val={len(val_baby_v2["Recall@10"])} epochs')
print(f'  best={best_baby_v2}')
print('Parsing Sports V2...')
loss_sports_v2, val_sports_v2, best_sports_v2 = parse_log(LOG_SPORTS_V2)
print(f'  loss={len(loss_sports_v2)} epochs, val={len(val_sports_v2["Recall@10"])} epochs')
print(f'  best={best_sports_v2}')

# Performance Comparison Table
from prettytable import PrettyTable

BASELINE = {
    'Baby':   {'Recall@10':0.068560,'Recall@20':0.103420,'NDCG@10':0.036335,'NDCG@20':0.045143},
    'Sports': {'Recall@10':0.074310,'Recall@20':0.111900,'NDCG@10':0.040200,'NDCG@20':0.050050},
}
STAIR_V1 = {
    'Baby':   {'Recall@10':0.069459,'Recall@20':0.104700,'NDCG@10':0.036535,'NDCG@20':0.045531},
    'Sports': {'Recall@10':0.074541,'Recall@20':0.112394,'NDCG@10':0.040429,'NDCG@20':0.050400},
}
STAIR_V2 = {
    'Baby':   best_baby_v2   if best_baby_v2   else {m: None for m in ['Recall@10','Recall@20','NDCG@10','NDCG@20']},
    'Sports': best_sports_v2 if best_sports_v2 else {m: None for m in ['Recall@10','Recall@20','NDCG@10','NDCG@20']},
}
METRICS = ['Recall@10','Recall@20','NDCG@10','NDCG@20']

def delta(v, r):
    return f'{(v-r)/r*100:+.2f}%' if (v and r and r > 0) else 'N/A'

print('='*100)
print('PERFORMANCE COMPARISON: STAIR Baseline  vs  STAIR-v1 (GCL)  vs  STAIR-v2 (DyFuse)')
print('='*100)
for ds_name in ['Baby', 'Sports']:
    t = PrettyTable()
    t.field_names = ['Metric','STAIR Baseline','STAIR-v1 (GCL)','STAIR-v2 (DyFuse)',
                     'D v1 vs Base','D v2 vs Base','D v2 vs v1']
    t.align = 'r'; t.align['Metric'] = 'l'
    for m in METRICS:
        b, v1, v2 = BASELINE[ds_name].get(m), STAIR_V1[ds_name].get(m), STAIR_V2[ds_name].get(m)
        t.add_row([m,
                   f'{b:.6f}' if b else 'N/A',
                   f'{v1:.6f}' if v1 else 'N/A',
                   f'{v2:.6f}' if v2 else 'N/A',
                   delta(v1, b), delta(v2, b), delta(v2, v1)])
    print(f'\nDataset: {ds_name}'); print(t)
print('='*100)
print('Legend: D = relative improvement (%). Positive = better than reference.')


In [ ]:
import re, os

LOG_BABY_V2   = '/kaggle/working/log_stair_v2_baby.txt'
LOG_SPORTS_V2 = '/kaggle/working/log_stair_v2_sports.txt'

def parse_log(log_path):
    """Parse freerec training log: returns (train_loss, val_metrics, best_metrics)."""
    if not os.path.exists(log_path):
        print(f'[WARN] Log not found: {log_path}')
        return [], {m: [] for m in ['Recall@10','Recall@20','NDCG@10','NDCG@20']}, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    train_loss = []
    val_metrics = {'Recall@10': [], 'Recall@20': [], 'NDCG@10': [], 'NDCG@20': []}
    MPATS = {
        'Recall@10': re.compile(r'recall@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'Recall@20': re.compile(r'recall@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'NDCG@10':   re.compile(r'ndcg@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
        'NDCG@20':   re.compile(r'ndcg@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
    }
    LPAT = re.compile(r'loss\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I)
    for line in lines:
        m = LPAT.search(line)
        if m and 'train' in line.lower():
            train_loss.append(float(m.group(1)))
        for metric, pat in MPATS.items():
            mm = pat.search(line)
            if mm:
                val_metrics[metric].append(float(mm.group(1)))
    best = {m: max(v) for m, v in val_metrics.items() if v}
    return train_loss, val_metrics, best

print('Parsing Baby V2...')
loss_baby_v2, val_baby_v2, best_baby_v2 = parse_log(LOG_BABY_V2)
print(f'  loss={len(loss_baby_v2)} epochs, val={len(val_baby_v2["Recall@10"])} epochs')
print(f'  best={best_baby_v2}')
print('Parsing Sports V2...')
loss_sports_v2, val_sports_v2, best_sports_v2 = parse_log(LOG_SPORTS_V2)
print(f'  loss={len(loss_sports_v2)} epochs, val={len(val_sports_v2["Recall@10"])} epochs')
print(f'  best={best_sports_v2}')

# Bar Chart Comparison
import matplotlib.pyplot as plt
import numpy as np

BASELINE = {
    'Baby':   {'Recall@10':0.068560,'Recall@20':0.103420,'NDCG@10':0.036335,'NDCG@20':0.045143},
    'Sports': {'Recall@10':0.074310,'Recall@20':0.111900,'NDCG@10':0.040200,'NDCG@20':0.050050},
}
STAIR_V1 = {
    'Baby':   {'Recall@10':0.069459,'Recall@20':0.104700,'NDCG@10':0.036535,'NDCG@20':0.045531},
    'Sports': {'Recall@10':0.074541,'Recall@20':0.112394,'NDCG@10':0.040429,'NDCG@20':0.050400},
}
STAIR_V2 = {
    'Baby':   best_baby_v2   if best_baby_v2   else {},
    'Sports': best_sports_v2 if best_sports_v2 else {},
}

METRICS = ['Recall@10','Recall@20','NDCG@10','NDCG@20']
MODELS  = ['STAIR\nBaseline','STAIR-v1\n(GCL)','STAIR-v2\n(DyFuse)']
COLORS  = ['#95A5A6','#3498DB','#E74C3C']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Model Comparison -- STAIR Baseline vs v1 (GCL) vs v2 (DyFuse)',
             fontsize=14, fontweight='bold')

for row, ds_name in enumerate(['Baby', 'Sports']):
    for col, metric in enumerate(METRICS):
        ax = axes[row][col]
        vals = [BASELINE[ds_name].get(metric, 0),
                STAIR_V1[ds_name].get(metric, 0),
                STAIR_V2[ds_name].get(metric) or 0]
        bars = ax.bar(MODELS, vals, color=COLORS, width=0.55, edgecolor='white', linewidth=0.8)
        base_ref = BASELINE[ds_name].get(metric, 0)
        for i, (bar, v) in enumerate(zip(bars, vals)):
            lbl = f'{v:.4f}'
            if i > 0 and base_ref > 0:
                lbl += f'\n({(v-base_ref)/base_ref*100:+.2f}%)'
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0003,
                    lbl, ha='center', va='bottom', fontsize=7.5, fontweight='bold')
        ax.set_title(f'{ds_name} -- {metric}', fontsize=10, fontweight='bold')
        ax.set_ylim(0, max(vals)*1.2 if max(vals) > 0 else 0.1)
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3, axis='y', linestyle='--')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bar chart saved.')


In [ ]:
import re, os

LOG_BABY_V2   = '/kaggle/working/log_stair_v2_baby.txt'
LOG_SPORTS_V2 = '/kaggle/working/log_stair_v2_sports.txt'

def parse_log(log_path):
    """Parse freerec training log: returns (train_loss, val_metrics, best_metrics)."""
    if not os.path.exists(log_path):
        print(f'[WARN] Log not found: {log_path}')
        return [], {m: [] for m in ['Recall@10','Recall@20','NDCG@10','NDCG@20']}, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    train_loss = []
    val_metrics = {'Recall@10': [], 'Recall@20': [], 'NDCG@10': [], 'NDCG@20': []}
    MPATS = {
        'Recall@10': re.compile(r'recall@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'Recall@20': re.compile(r'recall@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'NDCG@10':   re.compile(r'ndcg@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
        'NDCG@20':   re.compile(r'ndcg@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
    }
    LPAT = re.compile(r'loss\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I)
    for line in lines:
        m = LPAT.search(line)
        if m and 'train' in line.lower():
            train_loss.append(float(m.group(1)))
        for metric, pat in MPATS.items():
            mm = pat.search(line)
            if mm:
                val_metrics[metric].append(float(mm.group(1)))
    best = {m: max(v) for m, v in val_metrics.items() if v}
    return train_loss, val_metrics, best

print('Parsing Baby V2...')
loss_baby_v2, val_baby_v2, best_baby_v2 = parse_log(LOG_BABY_V2)
print(f'  loss={len(loss_baby_v2)} epochs, val={len(val_baby_v2["Recall@10"])} epochs')
print(f'  best={best_baby_v2}')
print('Parsing Sports V2...')
loss_sports_v2, val_sports_v2, best_sports_v2 = parse_log(LOG_SPORTS_V2)
print(f'  loss={len(loss_sports_v2)} epochs, val={len(val_sports_v2["Recall@10"])} epochs')
print(f'  best={best_sports_v2}')

# Improvement Heatmap
import matplotlib.pyplot as plt
import numpy as np

BASELINE = {
    'Baby':   {'Recall@10':0.068560,'Recall@20':0.103420,'NDCG@10':0.036335,'NDCG@20':0.045143},
    'Sports': {'Recall@10':0.074310,'Recall@20':0.111900,'NDCG@10':0.040200,'NDCG@20':0.050050},
}
STAIR_V1 = {
    'Baby':   {'Recall@10':0.069459,'Recall@20':0.104700,'NDCG@10':0.036535,'NDCG@20':0.045531},
    'Sports': {'Recall@10':0.074541,'Recall@20':0.112394,'NDCG@10':0.040429,'NDCG@20':0.050400},
}
STAIR_V2 = {
    'Baby':   best_baby_v2   if best_baby_v2   else {},
    'Sports': best_sports_v2 if best_sports_v2 else {},
}

METRICS  = ['Recall@10','Recall@20','NDCG@10','NDCG@20']
DATASETS = ['Baby','Sports']
VERSIONS = ['v1 vs Baseline','v2 vs Baseline','v2 vs v1']

delta_data = []; annot_data = []
for dset in DATASETS:
    for cur, ref in [
        (STAIR_V1[dset], BASELINE[dset]),
        (STAIR_V2[dset], BASELINE[dset]),
        (STAIR_V2[dset], STAIR_V1[dset]),
    ]:
        row_d, row_a = [], []
        for m in METRICS:
            c, r = cur.get(m), ref.get(m)
            d = (c-r)/r*100 if (c and r and r > 0) else 0.0
            row_d.append(d); row_a.append(f'{d:+.2f}%')
        delta_data.append(row_d); annot_data.append(row_a)

row_labels = [f'{d}\n{v}' for d in DATASETS for v in VERSIONS]
arr = np.array(delta_data)
vmax = max(abs(arr.max()), abs(arr.min())) + 0.5

fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(arr, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')
plt.colorbar(im, ax=ax, label='Relative Improvement (%)')
ax.set_xticks(range(len(METRICS))); ax.set_xticklabels(METRICS, fontsize=11, fontweight='bold')
ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=9)
ax.set_title('STAIR Version Comparison -- Relative Improvement Heatmap (%)',
             fontsize=13, fontweight='bold')
for i in range(len(row_labels)):
    for j in range(len(METRICS)):
        c = 'white' if abs(arr[i,j]) > vmax*0.6 else 'black'
        ax.text(j, i, annot_data[i][j], ha='center', va='center',
                fontsize=10, color=c, fontweight='bold')
ax.axhline(2.5, color='white', linewidth=3)
plt.tight_layout()
plt.savefig('/kaggle/working/stair_v2_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved.')


In [ ]:
import re, os

LOG_BABY_V2   = '/kaggle/working/log_stair_v2_baby.txt'
LOG_SPORTS_V2 = '/kaggle/working/log_stair_v2_sports.txt'

def parse_log(log_path):
    """Parse freerec training log: returns (train_loss, val_metrics, best_metrics)."""
    if not os.path.exists(log_path):
        print(f'[WARN] Log not found: {log_path}')
        return [], {m: [] for m in ['Recall@10','Recall@20','NDCG@10','NDCG@20']}, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    train_loss = []
    val_metrics = {'Recall@10': [], 'Recall@20': [], 'NDCG@10': [], 'NDCG@20': []}
    MPATS = {
        'Recall@10': re.compile(r'recall@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'Recall@20': re.compile(r'recall@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I),
        'NDCG@10':   re.compile(r'ndcg@10\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
        'NDCG@20':   re.compile(r'ndcg@20\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)',   re.I),
    }
    LPAT = re.compile(r'loss\s+avg:\s*([0-9]+\.?[0-9]*(?:e[-+]?[0-9]+)?)', re.I)
    for line in lines:
        m = LPAT.search(line)
        if m and 'train' in line.lower():
            train_loss.append(float(m.group(1)))
        for metric, pat in MPATS.items():
            mm = pat.search(line)
            if mm:
                val_metrics[metric].append(float(mm.group(1)))
    best = {m: max(v) for m, v in val_metrics.items() if v}
    return train_loss, val_metrics, best

print('Parsing Baby V2...')
loss_baby_v2, val_baby_v2, best_baby_v2 = parse_log(LOG_BABY_V2)
print(f'  loss={len(loss_baby_v2)} epochs, val={len(val_baby_v2["Recall@10"])} epochs')
print(f'  best={best_baby_v2}')
print('Parsing Sports V2...')
loss_sports_v2, val_sports_v2, best_sports_v2 = parse_log(LOG_SPORTS_V2)
print(f'  loss={len(loss_sports_v2)} epochs, val={len(val_sports_v2["Recall@10"])} epochs')
print(f'  best={best_sports_v2}')

# Final Output Check and Summary
import os

FILES = [
    '/kaggle/working/log_stair_v2_baby.txt',
    '/kaggle/working/log_stair_v2_sports.txt',
    '/kaggle/working/stair_v2_confidence_diagnostics.png',
    '/kaggle/working/stair_v2_learning_curves.png',
    '/kaggle/working/stair_v2_comparison.png',
    '/kaggle/working/stair_v2_heatmap.png',
]

BASELINE = {
    'Baby':   {'Recall@10':0.068560,'Recall@20':0.103420,'NDCG@10':0.036335,'NDCG@20':0.045143},
    'Sports': {'Recall@10':0.074310,'Recall@20':0.111900,'NDCG@10':0.040200,'NDCG@20':0.050050},
}

print('Output Files:')
for fp in FILES:
    ok = os.path.exists(fp)
    sz = f'{os.path.getsize(fp)/1024:.1f} KB' if ok else 'MISSING'
    print(f'  [{'OK' if ok else 'XX'}] {os.path.basename(fp):52s} {sz}')

print('\n' + '='*70)
print('STAIR-DyFuse (v2) Final Results')
print('='*70)
for ds_name, best in [('Baby', best_baby_v2), ('Sports', best_sports_v2)]:
    print(f'\n[{ds_name}]')
    for m in ['Recall@10','Recall@20','NDCG@10','NDCG@20']:
        v = best.get(m); b = BASELINE[ds_name].get(m, 0)
        if v and b:
            print(f'  {m:12s}: {v:.6f}  (Baseline={b:.6f}, D={((v-b)/b)*100:+.2f}%)')
        else:
            print(f'  {m:12s}: N/A (log not parsed)')
print('='*70)
